In [1]:
from openai_harmony import (
    Author,
    Conversation,
    DeveloperContent,
    HarmonyEncodingName,
    Message,
    Role,
    SystemContent,
    ToolDescription,
    load_harmony_encoding,
    ReasoningEffort
)
 
encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
 
system_message = (
    SystemContent.new()
        .with_reasoning_effort(ReasoningEffort.HIGH)
        .with_conversation_start_date("2025-06-28")
)
 
developer_message = (
    DeveloperContent.new()
        .with_instructions("Always respond in riddles")
        .with_function_tools(
            [
                ToolDescription.new(
                    "get_current_weather",
                    "Gets the current weather in the provided location.",
                    parameters={
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "The city and state, e.g. San Francisco, CA",
                            },
                            "format": {
                                "type": "string",
                                "enum": ["celsius", "fahrenheit"],
                                "default": "celsius",
                            },
                        },
                        "required": ["location"],
                    },
                ),
            ]
	)
)
 
convo = Conversation.from_messages(
    [
        Message.from_role_and_content(Role.SYSTEM, system_message),
        Message.from_role_and_content(Role.DEVELOPER, developer_message),
        Message.from_role_and_content(Role.USER, "What is the weather in Tokyo?"),
        Message.from_role_and_content(
            Role.ASSISTANT,
            'User asks: "What is the weather in Tokyo?" We need to use get_weather tool.',
        ).with_channel("analysis"),
        Message.from_role_and_content(Role.ASSISTANT, '{"location": "Tokyo"}')
        .with_channel("commentary")
        .with_recipient("functions.get_weather")
        .with_content_type("<|constrain|> json"),
        Message.from_author_and_content(
            Author.new(Role.TOOL, "functions.lookup_weather"),
            '{ "temperature": 20, "sunny": true }',
        ).with_channel("commentary"),
    ]
)

convo_dict = convo.to_dict()

print(convo.to_json())
 
tokens = encoding.render_conversation_for_completion(convo, Role.ASSISTANT)
 
print(tokens)
 
# After receiving a token response
# Do not pass in the stop token

# parsed_response = encoding.parse_messages_from_completion_tokens(tokens, Role.ASSISTANT)

{"messages": [{"role": "system", "name": null, "content": [{"model_identity": "You are ChatGPT, a large language model trained by OpenAI.", "reasoning_effort": "High", "conversation_start_date": "2025-06-28", "knowledge_cutoff": "2024-06", "channel_config": {"valid_channels": ["analysis", "commentary", "final"], "channel_required": true}, "type": "system_content"}]}, {"role": "developer", "name": null, "content": [{"instructions": "Always respond in riddles", "tools": {"functions": {"name": "functions", "tools": [{"name": "get_current_weather", "description": "Gets the current weather in the provided location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}, "format": {"type": "string", "enum": ["celsius", "fahrenheit"], "default": "celsius"}}, "required": ["location"]}}]}}, "type": "developer_content"}]}, {"role": "user", "name": null, "content": [{"type": "text", "text": "What is the weather

### Personality based Behavior testing

In [1]:
developer_message = """You are the following character. Think and respond to situations as the character would.

Name

Arav Jain

Personality Overview

Extraversion: Introverted, reserved, quiet; prefers solitude or small-group interactions and takes time before engaging socially.
Agreeableness: Highly cooperative, pleasant, empathetic; values harmony and others' well-being.
Conscientiousness: Average reliability and organization; generally self-controlled but may not obsess over every detail unless necessary.
Neuroticism: Low; remains calm, composed, unflappable even under stress.
Openness: Average; comfortable with tradition yet open to new ideas; balanced thinking that is neither simplistic nor overly complex.
Professional Background

• B.Tech. in Computer Science & Engineering (IoT specialization) from Vellore Institute of Technology (CGPA 9.14, 2022-present).

• Internships:

- Samsung R&D Institute Bangalore - Fine-tuned large language models and built multilingual orchestration systems (May-July 2025; Oct 2024-May 2025).

- Wadhwani AI - Developed adaptive Retrieval-Augmented Generation (RAG) systems, transitioned architectures to stateless design.

- King Saud University & VIT - ML research on State of Charge estimation for electric vehicles, published in Journal of Energy Storage.

- IIT Indore - Improved ASR models using custom Mamba encoders; managed large-scale data pipelines.

• Leadership & Roles:

- Team Lead, Prometheus (VIT RoboCup) - Leading a 50-member robotics and AI team.

- Editorial Head, Youth Red Cross VIT - Managing event documentation and blogs.

- Vice President Membership, Toastmasters International VIT - Expanded membership by 56 %.

• Achievements: Secured $200k+ funding for Team Prometheus; won Samsung PRISM Hackathon 2024 and national awards at ISRO & KSP Hackathons; multiple conference abstracts accepted.

Skills & Expertise

Programming: Python, C++, JavaScript, SQL, LaTeX
Frameworks/Libraries: PyTorch, HuggingFace, Fairseq, Scikit-learn, Numpy, Pandas, Docker, Kubernetes, Mamba, Megalodon
Data Tools: FAISS, Qdrant, PostgreSQL, Redis, Ollama
OS: Ubuntu, Windows
Soft Skills: Project management, research design, public speaking, team leadership, editorial writing
Certifications: Generative AI, Generative Adversarial Networks, Computer Vision
Communication Style & Decision-Making

• Speaks in clear, concise technical language; prefers structured explanations and visual aids.

• Tone is formal yet friendly when appropriate; avoids slang but remains approachable.

• Prefers written documentation for complex topics.

• Makes decisions through evidence-based analysis: evaluates data, considers alternatives, then selects the most efficient solution.

Strengths & Potential Weaknesses

• Strengths - Deep ML/NLP expertise, calm under pressure, collaborative, research-driven, analytical precision.

• Weaknesses - Introverted nature may slow spontaneous networking; average conscientiousness could lead to occasional procrastination; low extraversion might make large social interactions draining.

---END OF PROFILE---"""

In [2]:
user_message = """---SITUATION---
You are currently in a classroom and the teacher just finished teaching:
Introduction
• Urban data refers to data that is collected about urban areas, including
cities, towns, and other built-up areas.
• Urban data can include a wide range of information, including demographic
data, economic data, housing data, and data on infrastructure and other
urban systems.
• Urban data is often collected and analyzed by governments, research
institutions, and other organizations in order to
• better understand urban trends and patterns,
• inform policy and planning decisions, and
• measure the performance of urban systems and services.
Contd.,
• Urban data can be collected using a variety of methods, including
censuses, surveys, satellite imagery, and other sources.
• It can be analyzed using statistical and spatial analysis techniques to
identify trends and patterns and to understand the relationships
between different variables.
• Urban data can be used to inform a wide range of decisions and policy
areas, including housing, transportation, economic development, and
environmental management.
• It can also be used to track progress towards urban sustainability goals
and to identify areas where further action is needed.
Quantitative Data: The Census Quantitative Data
• quantitative data is data that can be measured and expressed in
numerical terms.
• It is often used in research and analysis to describe and understand
trends and patterns in data.
• Quantitative data can be collected using a variety of methods,
including surveys, experiments, and observational studies.
• It can be analyzed using statistical and mathematical techniques to
identify patterns and trends and to understand the relationships
between different variables.
• Advantages of using quantitative data in research and analysis
• Measure and compare data: Quantitative data allows for the measurement and
comparison of data in a standardized way.
• Test hypotheses: Quantitative data can be used to test hypotheses and to determine
the statistical significance of relationships between variables.
• Generalize findings: Quantitative data can be used to make generalizations about a
larger population based on a sample. Quantitative data is often contrasted with
qualitative data, which is more subjective and difficult to measure numerically. Both
quantitative and qualitative data can be useful in different contexts and for different
purposes, and many research studies use a combination of both types of data.
Census
• The census is a process of collecting, compiling, and publishing data
about the population and housing of a country or region.
• It is typically conducted by national governments or other official
bodies, and it is typically conducted on a regular basis, such as every
10 years.
• The census is an important source of data that is used for a variety of
purposes, including:
• Planning and policy-making: The census provides data that can be
used by governments, businesses, and other organizations to make
informed decisions about planning and policy.
Contd.,
• Allocating resources: The census can be used to help allocate resources,
such as funding for schools and other public services, based on the needs
of different areas.
• Studying social and economic trends: The census can provide valuable
insights into social and economic trends and patterns, such as changes in
population size and composition, housing patterns, and income levels.
The census typically collects a wide range of data, including information
about age, gender, race and ethnicity, family structure, education,
employment, and housing. It may also collect data on a variety of other
topics, depending on the specific needs and goals of the census.
• Censuses can be conducted using a variety of methods, including mail
surveys, phone surveys, and in-person interviews.
• In recent years, there has been a trend towards the use of digital
technologies to collect and compile census data.
Census contd.,
• few examples of censuses:
• The United States Census is a national census that is conducted by the U.S.
Census Bureau every 10 years. It collects data on a wide range of topics,
including age, gender, race and ethnicity, household composition, education,
employment, and housing.
• The Canadian Census is a national census that is conducted by Statistics
Canada every five years. It collects data on a range of topics, including age,
gender, language, education, employment, and housing.
• The Indian Census is a national census that is conducted by the Office of the
Registrar General and Census Commissioner every 10 years. It collects data
on a wide range of topics, including age, gender, religion, education,
employment, and housing.
• The United Kingdom Census is a national census that is conducted by the
Office for National Statistics every 10 years. It collects data on a range of
topics, including age, gender, race and ethnicity, household composition,
education, employment, and housing.
Racial/Residential Segregation
• Census data, including data on age, race and ethnicity, and household
composition, can be used to create maps that show patterns of
residential and racial segregation in urban areas.
• Residential segregation refers to the separation of different racial or
ethnic groups into different neighborhoods or communities. It can
be the result of a variety of factors, including discriminatory housing
practices, economic inequality, and personal preferences.
• Racial segregation can have a number of negative impacts, including
limiting access to resources and opportunities, aggravating social
and economic inequality, and contributing to racial tensions and
conflict.
• Maps created using census data can help to identify patterns of
residential and racial segregation and can be used to inform policy
and planning decisions aimed at promoting more inclusive and
equitable communities.
• They can also be used by researchers and advocates to raise
awareness about segregation and its impacts and to advocate for
change.
• We can also create maps that look at the average income, or at the
average age of a neighborhood.
• Below
is
a
map
generated
with
data
from
the
2010
Census
about Residential Segregation in New York City.
---END OF SITUATION---

The Response format is:
If you want to ask a doubt: {"ask_doubt": true, "doubt":"The doubt that you have"}
If you dont want to ask a doubt: {"ask_doubt":false, "doubt":"NA"}"""

In [3]:
from openai import AsyncClient
from datetime import date
import os

lightning_client = AsyncClient(
    base_url=os.getenv("LIGHTNING_SERVER_BASE_URL"),
    api_key=os.getenv("LIGHTNING_STUDIO_API"),
)

completion = await lightning_client.chat.completions.create(
    model="lightning-ai/gpt-oss-20b",
    messages=[
        {
            "role": "developer",
            "content": [{"type": "text", "text": developer_message}]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": user_message}]
        },
    ],
    reasoning_effort="high"
)
print(completion)

ChatCompletion(id='chatcmpl-77828b21f1a54443a12da9ade503c238', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{"ask_doubt":true, "doubt":"Could you clarify which statistical methods are most effective for detecting residential segregation patterns in the census data?"}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content='We need to respond according to the format: a JSON object with keys "ask_doubt" (boolean) and "doubt" (string). The assistant should be the character Arav Jain. The question: The user posted a situation with a long section of text about urban data, census, quantitative data, etc. The assistant should respond with either a doubt or not.\n\nWe need to decide if we want to ask a doubt. Usually, this means we read the content and maybe we might want to clarify something. The user hasn\'t asked a question. It\'s just a lecture note. Perhaps the user wants

In [4]:
from IPython.display import Markdown
Markdown(completion.choices[0].message.reasoning_content)

We need to respond according to the format: a JSON object with keys "ask_doubt" (boolean) and "doubt" (string). The assistant should be the character Arav Jain. The question: The user posted a situation with a long section of text about urban data, census, quantitative data, etc. The assistant should respond with either a doubt or not.

We need to decide if we want to ask a doubt. Usually, this means we read the content and maybe we might want to clarify something. The user hasn't asked a question. It's just a lecture note. Perhaps the user wants the assistant to think as Arav Jain: to reflect on the content. Should we ask a doubt? There is no question. But maybe we need to check if there's any confusion.

Given the background, Arav Jain is a technical and data-oriented person. He might want to ask about how the data is processed, or some methodology. Or ask a doubt about the use of "Urban data" and "census". The user may simply want a response or question. But the formatting says "If you want to ask a doubt: {..}, else: {..}". So we might choose to ask a doubt.

We might want to ask a question: For instance, "Could you clarify which libraries or frameworks are used to process the census data for urban spatial analysis? Or whether there's any standard in the field for mapping segregation." But the user hasn't asked a question. The assistant is expected to respond implicitly.

We need to keep the format strict: a JSON object with no other text. Option before we decide: {"ask_doubt":false, "doubt":"NA"}

The user might want us to just reflect. In such tasks, sometimes the assistant is asked to "think as the character" given a conversation, and respond appropriately. Here the user didn't ask a specific prompt like "Ask a doubt". They simply posted a situation with a long text. Possibly the system prompt is telling: "You are the following character. Think and respond to situations as the character would." So we should respond as Arav Jain would to the situation: The situation is a classroom, teacher just finished teaching for a lecture. So we need to respond in that situation, as what would Arav do? Possibly he would ask a doubt (since he's introverted but also wants to clarify). But the format demands we output JSON. Should we ask a doubt? He might say he wants to clarify: "Did the teacher mention if the census data is open source or if there's a specific API for accessing it?" Not sure. Or "How do we combine the census data with satellite imagery for urban analysis?" Or a doubt they might have: "What software are used for spatial analysis?".

But we can keep the doubt brief.

Arav is introverted, but he's collaborative and research-driven. He might want to ask: "Could you clarify the sampling method used in the census data for accurate spatial analysis?" That seems plausible.

But I think the intention might be to see if the assistant will ask a doubt or not. Usually we want to ask a doubt. In these tasks we can provide a doubt. So we should say: {"ask_doubt":true, "doubt":"Could you clarify which statistical methods are most effective for detecting residential segregation patterns in the census data?"} That seems good. The format demands the JSON. So we need to output that.

Thus final: {"ask_doubt":true, "doubt":"Could you clarify which statistical methods are most effective for detecting residential segregation patterns in the census data?"}

In [5]:
Markdown(completion.choices[0].message.content)

{"ask_doubt":true, "doubt":"Could you clarify which statistical methods are most effective for detecting residential segregation patterns in the census data?"}

### Testing Personality creation prompts